
# Guía práctica — Clasificación, Detección y Segmentación en Computer Vision

## Módulo
Computer Vision — Deep Learning aplicado a imagen

---

# Objetivo de esta guía

Esta guía tiene como objetivo introducir las tres grandes tareas de Deep Learning en visión por computador:

1. Clasificación.
2. Detección de objetos.
3. Segmentación.

La idea no es solo ejecutar modelos, sino comprender:

- Qué hace cada tarea.
- Qué arquitectura suele utilizarse.
- Qué entrada y salida produce cada modelo.
- Qué métricas se usan.
- Qué problemas aparecen.
- Qué modificaciones pueden hacerse.

Además, se incluyen ejemplos simples y progresivos usando TensorFlow/Keras y modelos modernos.



# 1. Diferencia conceptual entre clasificación, detección y segmentación



| Tarea | Objetivo | Salida esperada |
|---|---|---|
| Clasificación | Determinar qué aparece en una imagen | Etiqueta |
| Detección | Localizar objetos y clasificarlos | Bounding boxes + etiquetas |
| Segmentación | Delimitar píxel a píxel los objetos | Máscara |



# 2. Clasificación de imágenes

La clasificación intenta responder:

> “¿Qué aparece en esta imagen?”

Puede ser:

- Binaria.
- Multiclase.
- Multilabel.



# 2.1 Clasificación binaria — Cats vs Dogs


In [ ]:

import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models


In [ ]:

(ds_train, ds_test), ds_info = tfds.load(
    "cats_vs_dogs",
    split=["train[:80%]", "train[80%:]"],
    as_supervised=True,
    with_info=True
)


In [ ]:

plt.figure(figsize=(10, 5))

for i, (image, label) in enumerate(ds_train.take(6)):
    plt.subplot(2, 3, i + 1)
    plt.imshow(image)
    plt.title("dog" if label.numpy() else "cat")
    plt.axis("off")

plt.show()


In [ ]:

IMG_SIZE = 128
BATCH_SIZE = 32

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image / 255.0
    return image, label


In [ ]:

train_ds = (
    ds_train
    .map(preprocess)
    .shuffle(1000)
    .batch(BATCH_SIZE)
)

test_ds = (
    ds_test
    .map(preprocess)
    .batch(BATCH_SIZE)
)



# Estructura de una CNN simple


In [ ]:

model = models.Sequential([
    layers.Conv2D(32, 3, activation="relu", input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Flatten(),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),

    layers.Dense(1, activation="sigmoid")
])



## Claves importantes

### Última capa

```python
Dense(1, activation="sigmoid")
```

Porque:

- solo hay 2 clases.
- queremos una probabilidad entre 0 y 1.


In [ ]:

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [ ]:

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10
)



# Modificaciones propuestas

## Modificación 1

Añadir BatchNormalization.

```python
layers.BatchNormalization()
```

## Modificación 2

Añadir Data Augmentation.

```python
layers.RandomFlip("horizontal")
layers.RandomRotation(0.1)
```

## Modificación 3

Usar Transfer Learning.

- MobileNetV2.
- ResNet50.
- EfficientNet.



# 2.2 Clasificación multiclase — CIFAR-10


In [ ]:

from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()


In [ ]:

class_names = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
]


In [ ]:

X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0


In [ ]:

model = models.Sequential([
    layers.Conv2D(32, 3, activation="relu", input_shape=(32, 32, 3)),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Flatten(),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),

    layers.Dense(10, activation="softmax")
])



## Diferencias importantes con clasificación binaria

### Última capa

```python
Dense(10, activation="softmax")
```

Porque:

- hay 10 clases.
- queremos distribución de probabilidades.


In [ ]:

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)



# Ejercicios propuestos

1. Añadir BatchNormalization.
2. Cambiar kernels de (3,3) a (5,5).
3. Probar ResNet50 y MobileNetV2.



# 3. Detección de objetos

La detección intenta responder:

- ¿Qué objetos hay?
- ¿Dónde están?



# 3.1 Detección simple con YOLO


In [ ]:

# !pip install ultralytics


In [ ]:

from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt


In [ ]:

model = YOLO("yolov8n.pt")


In [ ]:

results = model("imagen.jpg")


In [ ]:

result_image = results[0].plot()

plt.figure(figsize=(10,10))
plt.imshow(cv2.cvtColor(result_image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()



# Qué devuelve YOLO

## Bounding boxes

```python
results[0].boxes
```

## Clases detectadas

```python
results[0].boxes.cls
```

## Confianza

```python
results[0].boxes.conf
```



# Conceptos importantes

## IoU

Intersection over Union.

## NMS

Non-Maximum Suppression.

## mAP

Mean Average Precision.



# Ejercicios de detección

1. Comparar yolov8n, yolov8s y yolov8m.
2. Cambiar el threshold de confianza.
3. Detectar usando webcam.
4. Detectar múltiples objetos en escenas urbanas.



# 4. Segmentación

La segmentación asigna una clase a cada píxel.

Salida:

```python
mask[y, x] = class_id
```



# Tipos de segmentación

## Segmentación semántica

Todos los píxeles de la misma clase tienen misma etiqueta.

## Segmentación de instancias

Cada objeto tiene identidad propia.



# Ejemplo conceptual de segmentación


In [ ]:

import numpy as np
import matplotlib.pyplot as plt


In [ ]:

image = np.random.rand(256, 256)

mask = image > 0.5


In [ ]:

plt.figure(figsize=(8,8))
plt.imshow(mask, cmap="gray")
plt.title("Máscara segmentada")
plt.axis("off")
plt.show()



# Problemas habituales

- Bordes incorrectos.
- Objetos pequeños.
- Objetos solapados.
- Máscaras ruidosas.



# Ejercicios de segmentación

1. Cambiar umbral de segmentación.
2. Aplicar erosión y dilatación.
3. Comparar máscaras antes y después del postprocesado.



# 4.1 Segmentación con SAM

SAM = Segment Anything Model.

Modelo de Meta que permite segmentar objetos usando:

- puntos.
- cajas.
- prompts.


In [ ]:

# !pip install segment-anything


In [ ]:

from segment_anything import sam_model_registry, SamPredictor
import numpy as np
import cv2


In [ ]:

sam = sam_model_registry["vit_b"](
    checkpoint="sam_vit_b.pth"
)


In [ ]:

predictor = SamPredictor(sam)


In [ ]:

image = cv2.imread("image.jpg")
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

predictor.set_image(image)


In [ ]:

input_point = np.array([[500, 300]])
input_label = np.array([1])


In [ ]:

masks, scores, logits = predictor.predict(
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=True
)


In [ ]:

plt.figure(figsize=(8,8))
plt.imshow(image)
plt.imshow(masks[0], alpha=0.5)
plt.axis("off")
plt.show()



# Qué hace SAM realmente

SAM no clasifica.

SAM segmenta regiones coherentes.



# Ejercicios con SAM

1. Cambiar el punto de entrada.
2. Usar múltiples puntos.
3. Usar bounding boxes como prompt.
4. Comparar máscaras obtenidas.



# 5. Comparativa final

| Tarea | Salida | Modelos típicos |
|---|---|---|
| Clasificación | Etiqueta | CNN, ResNet, EfficientNet |
| Detección | Bounding boxes | YOLO, Faster R-CNN, DETR |
| Segmentación | Máscaras | U-Net, DeepLab, SAM |



# 6. Preguntas finales

1. ¿Qué diferencia hay entre clasificación y detección?
2. ¿Qué diferencia hay entre detección y segmentación?
3. ¿Por qué YOLO es rápido?
4. ¿Qué problema intenta resolver NMS?
5. ¿Qué representa IoU?
6. ¿Qué diferencia hay entre segmentación semántica y segmentación de instancias?
7. ¿Qué ventaja tiene SAM?
8. ¿Qué limitaciones tiene SAM?



# 7. Ideas de ampliación

## Clasificación

- Transfer Learning.
- Grad-CAM.
- Multilabel.
- Ensembles.

## Detección

- Entrenar YOLO con dataset propio.
- Roboflow.
- Tracking.
- DeepSORT.

## Segmentación

- U-Net completo.
- Mask R-CNN.
- Segmentación médica.
- Segmentación en vídeo.



# Conclusión

Las tres tareas principales de visión por computador tienen objetivos distintos:

- Clasificación → “qué hay”.
- Detección → “qué hay y dónde”.
- Segmentación → “qué pertenece exactamente a cada objeto”.

La elección del modelo depende:

- del problema.
- del dataset.
- de la precisión deseada.
- de los recursos computacionales.
- de la velocidad requerida.
